# CNN · остаточная сеть и красное смещение

Исследовательский вариант остаточной CNN с отдельной веткой красного смещения. Основные сопоставляемые запуски находятся в `notebooks/training/`.

Пути определяются автоматически от корня репозитория. См. [воспроизведение](../../docs/reproduction.md) и [описание результатов](../../docs/results.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Импорты


In [ ]:
import numpy as np
import sys
import os
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import pandas as pd

## Воспроизводимость


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

## Окружение и параметры


In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 128

## Архитектура модели


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

## Архитектура модели


In [ ]:
class DeepResNetWithScalar(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(ResidualBlock, 64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(ResidualBlock, 64, 128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(ResidualBlock, 128, 256, num_blocks=3, stride=2)
        self.layer4 = self._make_layer(ResidualBlock, 256, 512, num_blocks=3, stride=2)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.redshift_fc = nn.Sequential(
            nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(512 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def _make_layer(self, block, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(block(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(block(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, images, redshift):
        x = self.stem(images)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)

        z = self.redshift_fc(redshift)

        combined = torch.cat((x, z), dim=1)

        out = self.fc(combined)
        return out

## Модель и оптимизатор


In [ ]:
model = DeepResNetWithScalar().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = torch.nn.MSELoss()

In [ ]:
# train_model(model, train_loader, val_loader, num_epochs=EPOCHS)

## Загрузка данных


In [ ]:
class ClusterDataset(Dataset):
    def __init__(self, images_dir, indices):
        self.images_dir = images_dir
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        index = self.indices[idx]
        img_path = os.path.join(self.images_dir, f"{index}.npy")
        image = np.load(img_path).astype(np.float32)
        csv_path = os.path.join(
            str(PROJECT_ROOT / "data/raw"),
            "clusters_selected.csv",
        )
        df = pd.read_csv(csv_path)
        row = df.iloc[index]
        redshift = np.array([row["redshift"]], dtype=np.float32)
        target = np.float32(row["M500cUncorr"])
        return torch.tensor(image), torch.tensor(redshift), torch.tensor(target)

## Подготовка выборок


In [ ]:
train_indices = np.load(PROJECT_ROOT / "data/splits/raw/train_indices.npy")
test_indices = np.load(PROJECT_ROOT / "data/splits/raw/test_indices.npy")

train_dataset = ClusterDataset(
    str(PROJECT_ROOT / "data/processed/images"), train_indices
)
test_dataset = ClusterDataset(str(PROJECT_ROOT / "data/processed/images"), test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

In [ ]:
model.train()
losses = []

for epoch in range(EPOCHS):
    start_t = time.time()
    total_loss = 0
    for img, scalars, target in train_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        # Проверка форм
        optimizer.zero_grad()
        pred = model(img, scalars)
        print(
            f"pred shape: {pred.shape}, target shape: {target.shape}"
        )  # Отладочный вывод
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    losses.append(total_loss / len(train_loader))
    scheduler.step()

    if epoch == 0:
        print("Epoch 1 time:", time.time() - start_t)

    if (epoch + 1) % 10 == 0:
        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for img, scalars, target in test_loader:
                img = img.to(device)
                scalars = scalars.to(device)
                target = target.to(device)

                pred = model(img, scalars)
                preds.append(pred)
                targets.append(target)

        model.train()

        preds = torch.cat(preds)
        targets = torch.cat(targets)

        mae = torch.abs(targets - preds)
        mean_mae = mae.mean().item()
        max_mae = mae.max().item()

        relative_errors = mae / torch.abs(targets)
        mean_rel_error = relative_errors.mean().item()
        max_rel_error = relative_errors.max().item()

        print(f"EPOCH={epoch+1}")
        print(f"Средняя MAE {mean_mae:.4f}")
        print(f"Максимальная MAE {max_mae:.4f}")
        print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
        print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

## Оценка и графики


In [ ]:
losses = np.asarray(losses)

model.eval()
preds = []
targets = []

with torch.no_grad():
    for img, scalars, target in test_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device)

        pred = model(img, scalars)

        preds.append(pred.cpu())
        targets.append(target.cpu())

preds = torch.cat(preds)
targets = torch.cat(targets)

mae = torch.abs(targets - preds)
mean_mae = mae.mean().item()
max_mae = mae.max().item()

relative_errors = mae / torch.abs(targets)
mean_rel_error = relative_errors.mean().item()
max_rel_error = relative_errors.max().item()

print(f"EPOCH={epoch+1}")
print(f"Средняя MAE: {mean_mae:.4f}")
print(f"Максимальная MAE: {max_mae:.4f}")
print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

output_dir = str(PROJECT_ROOT / "results/exploration/cnn_residual") + "/"
np.save(output_dir + "losses.npy", losses)
np.save(output_dir + "targets.npy", targets)
np.save(output_dir + "predictions.npy", preds)